In [8]:
import os
import zipfile
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
import sys
sys.path.append("/bohr/dataset-gdiz/v1")
from spectrogram_dataset import SpectrogramDataset

In [28]:
class AudioNet(nn.Module):
    def __init__(self):
        super().__init__()
        model = resnet18(weights=ResNet18_Weights.DEFAULT)
        old_conv = model.conv1
        model.conv1 = nn.Conv2d(1, 64,kernel_size=7,stride=2,padding=3,bias=False)
        with torch.no_grad():
            model.conv1.weight.copy_(
                old_conv.weight.mean(dim=1, keepdim=True)
            )
        model.fc = nn.Linear(model.fc.in_features, 2)
        self.model = model
    def forward(self, x):
        return self.model(x)

In [29]:
def train_one_epoch(model, train_loader, val_loader, criterion, optimizer, device):
    model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc="Train"):
        x = batch["spectrogram"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    print(f"Train Loss: {train_loss:.4f}")

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Val Split"):
            x = batch["spectrogram"].to(device)
            y = batch["label"].to(device)
            output = model(x)
            loss = criterion(output, y)

            val_loss += loss.item()

    val_loss /= len(val_loader)
    print(f"Val Split Loss: {val_loss:.4f}")

In [30]:
def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Test"):
            x = batch["spectrogram"].to(device)
            output = model(x)
            pred = torch.argmax(output, dim=1)
            preds.extend(pred.cpu().numpy())
    return preds

In [31]:
def save_submission_csv(preds, save_name):
    df = pd.DataFrame(preds)
    df.to_csv(save_name, index=False, header=False)

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

In [34]:
print(device)

In [36]:
# full_train_set = SpectrogramDataset("/bohr/dataset-gdiz/v1/training_set/")

# val_size = int(0.2 * len(full_train_set))
# train_size = len(full_train_set) - val_size
# train_set, val_split_set = random_split(full_train_set, [train_size, val_size])

# train_loader = DataLoader(train_set, batch_size=32)
# val_split_loader = DataLoader(val_split_set, batch_size=32)

# train_one_epoch(model, train_loader, val_split_loader, criterion, optimizer, device)

In [ ]:
full_train_set = SpectrogramDataset("/bohr/dataset-gdiz/v1/training_set/")
train_dl = DataLoader(full_train_set, batch_size = 32, shuffle= True, num_workers= 4, pin_memory =True)

opt = torch.optim.AdamW(model.parameters(), lr = 1e-3, weight_decay = 2e-5)
loss_fn = nn.CrossEntropyLoss()

for e in range(5):
    train_one_epoch(model, train_dl, train_dl, loss_fn, opt, device)

## 测试阶段

In [21]:
## 获取用于AB榜评测的验证集与测试集
## 【仅在notebook提交至比赛后可正常获取】
## 【代码调试阶段报错是正常现象，因为以下数据不对选手公开，无法在这个阶段被读取】

if os.environ.get('ANSWER_PATH'):
    PATH = os.environ.get("ANSWER_PATH") + "/" 
else:
    print("Baseline运行时，因为无法读取测试集，所以后续会有报错，属于正常现象")  

In [19]:
val_set = SpectrogramDataset(PATH+"/validation_set/")
test_set = SpectrogramDataset(PATH+"/testing_set/")

val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

val_preds = predict(model, val_loader, device)
test_preds = predict(model, test_loader, device)

In [20]:
save_submission_csv(val_preds, "submissionA.csv")
save_submission_csv(test_preds, "submissionB.csv")
with zipfile.ZipFile("submission.zip", "w") as zipf:
    zipf.write("submissionA.csv")
    zipf.write("submissionB.csv")
os.remove("submissionA.csv")
os.remove("submissionB.csv")